In [17]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

# Today
today = datetime.today().strftime('%Y-%m-%d')

# Create output directories
for directory in ["models", "test_set_predictions", "training_loss", "metrics"]:
    os.makedirs(directory, exist_ok=True)

top_10_symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B', 'UNH', 'JPM']

def create_sequences(data, time_steps=60):
    X, y = [], []
    for i in range(time_steps, len(data)):
        X.append(data[i - time_steps:i])
        y.append(data[i][0])  # 'Close' price
    return np.array(X), np.array(y)

def inverse_close(preds, scaler):
    combined = np.concatenate([preds, np.zeros_like(preds)], axis=1)
    return scaler.inverse_transform(combined)[:, 0]

def print_metrics(true_vals, preds):
    mae = mean_absolute_error(true_vals, preds)
    mse = mean_squared_error(true_vals, preds)
    r2 = r2_score(true_vals, preds)
    return mae, mse, r2

for symbol in top_10_symbols:
    try:
        df = pd.read_csv(f"../LSTM/datasets/{symbol}_daily_data.csv")
        df = df.iloc[1:].reset_index(drop=True)
        df.to_csv(f"../LSTM/datasets/{symbol}_daily_data.csv", index=False)

        print(f"\n=== Processing {symbol} ===")

        # === Load Data ===
        stock_df = pd.read_csv(f"../LSTM/datasets/{symbol}_daily_data.csv")
        stock_df['Date'] = pd.to_datetime(stock_df['Date'])

        sentiment_df = pd.read_csv(f"../Sentiment-Analysis/sentiment/{today}/{symbol}_sentiment.csv")
        sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
        sentiment_df['sentiment_score'] = sentiment_df['sentiment'].map({
            'POSITIVE': 1, 'NEGATIVE': -1, 'NEUTRAL': 0
        })

        daily_sentiment = sentiment_df.groupby('date')['sentiment_score'].mean().reset_index()
        daily_sentiment.columns = ['Date', 'Sentiment']

        merged_df = pd.merge(stock_df[['Date', 'Close']], daily_sentiment, on='Date', how='left')
        merged_df['Sentiment'].fillna(0, inplace=True)

        # === Scale Data ===
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(merged_df[['Close', 'Sentiment']])

        time_steps = 60
        X, y = create_sequences(scaled_data, time_steps)

        split_index = int(len(X) * 0.8)
        X_train, y_train = X[:split_index], y[:split_index]
        X_test, y_test = X[split_index:], y[split_index:]

        # === Build Model ===
        model = Sequential([
            LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
            Dropout(0.2),
            LSTM(32),
            Dropout(0.2),
            Dense(1)
        ])
        model.compile(optimizer='adam', loss='mean_squared_error')

        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        checkpoint_path = f"models/{symbol}_best_model.h5"
        checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True)

        # === Train Model ===
        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=100,
            batch_size=32,
            callbacks=[early_stop, checkpoint],
            verbose=0
        )

        # === Predict ===
        train_preds = model.predict(X_train)
        test_preds = model.predict(X_test)

        train_prices = inverse_close(train_preds, scaler)
        test_prices = inverse_close(test_preds, scaler)
        true_train = inverse_close(y_train.reshape(-1, 1), scaler)
        true_test = inverse_close(y_test.reshape(-1, 1), scaler)

        # === Metrics ===
        train_mae, train_mse, train_r2 = print_metrics(true_train, train_prices)
        test_mae, test_mse, test_r2 = print_metrics(true_test, test_prices)

        # === Predict Next Day ===
        last_sequence = scaled_data[-time_steps:].reshape(1, time_steps, 2)
        next_scaled_pred = model.predict(last_sequence)
        next_day_price = inverse_close(next_scaled_pred, scaler)[0]

        # === Save Metrics to CSV ===
        metrics_df = pd.DataFrame([{
            'symbol': symbol,
            'train_mae': round(train_mae, 4),
            'train_mse': round(train_mse, 4),
            'train_r2': round(train_r2, 4),
            'test_mae': round(test_mae, 4),
            'test_mse': round(test_mse, 4),
            'test_r2': round(test_r2, 4),
            'next_day_prediction': round(next_day_price, 4)
        }])
        metrics_df.to_csv(f"metrics/{symbol}_metrics.csv", index=False)

        # === Plot Predictions ===
        plt.figure(figsize=(12, 6))
        plt.plot(true_test, label='Actual')
        plt.plot(test_prices, label='Predicted')
        plt.title(f"{symbol} - Stock Price Prediction (Test Set)")
        plt.xlabel("Time")
        plt.ylabel("Price")
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"test_set_predictions/{symbol}_price_plot.png")
        plt.close()

        # === Plot Loss ===
        plt.figure(figsize=(8, 4))
        plt.plot(history.history['loss'], label='Training Loss')
        plt.plot(history.history['val_loss'], label='Validation Loss')
        plt.title(f"{symbol} - Loss Curve")
        plt.xlabel("Epochs")
        plt.ylabel("MSE Loss")
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"training_loss/{symbol}_loss_plot.png")
        plt.close()

        print(f"✓ Completed {symbol}")

    except Exception as e:
        print(f"✗ Error processing {symbol}: {e}")



=== Processing AAPL ===


/tmp/ipykernel_13799/2849243493.py:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Sentiment'].fillna(0, inplace=True)
/workspaces/Stock-Market-Prediction-AI-Model/myenv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
✓ Completed AAPL

=== Processing MSFT ===


/tmp/ipykernel_13799/2849243493.py:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Sentiment'].fillna(0, inplace=True)
/workspaces/Stock-Market-Prediction-AI-Model/myenv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
✓ Completed MSFT

=== Processing GOOGL ===


...

159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
✓ Completed BRK-B

=== Processing UNH ===


/tmp/ipykernel_13799/2849243493.py:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Sentiment'].fillna(0, inplace=True)
/workspaces/Stock-Market-Prediction-AI-Model/myenv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
✓ Completed UNH

=== Processing JPM ===


/tmp/ipykernel_13799/2849243493.py:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Sentiment'].fillna(0, inplace=True)
/workspaces/Stock-Market-Prediction-AI-Model/myenv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
✓ Completed JPM
